In [3]:
#!pip install trl
#!pip install -U bitsandbytes>=0.46.1
#!pip install unsloth

In [2]:
import torch
if torch.cuda.is_available():
    print(f"GPU is active: {torch.cuda.get_device_name(0)}")
else:
    print("GPU NOT FOUND. Check Runtime settings.")

GPU is active: Tesla T4


In [1]:
import os
os.environ["UNSLOTH_DISABLE_COMPILE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import gc
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

gc.collect()
torch.cuda.empty_cache()

# --- MODEL ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=512,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
)

# --- DATA ---
dataset = load_dataset("argilla/farming", split="train")

def preprocess(example):
    text = f"""### Instruction:
{example['instruction']}

### Response:
{example['response']}"""

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
    )

    # IMPORTANT: explicit labels (no auto alignment bugs)
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(preprocess, remove_columns=dataset.column_names)

# --- COLLATOR ---
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

# --- TRAINER (RAW TRAINER, NOT SFTTrainer) ---
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
        fp16=torch.cuda.is_available(),
        report_to="none",
    ),
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/1695 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,695 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.437475
2,1.339166
3,2.121853
4,1.633792
5,1.532659
6,1.413654
7,1.295312
8,1.635958
9,1.769566
10,1.925144


Step,Training Loss
1,1.437475
2,1.339166
3,2.121853
4,1.633792
5,1.532659
6,1.413654
7,1.295312
8,1.635958
9,1.769566
10,1.925144


TrainOutput(global_step=60, training_loss=1.4999483625094097, metrics={'train_runtime': 960.9415, 'train_samples_per_second': 0.5, 'train_steps_per_second': 0.062, 'total_flos': 1.112830925340672e+16, 'train_loss': 1.4999483625094097, 'epoch': 0.2831858407079646})

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer # Optimized for this specific workflow

# 1. Configuration
model_id = "meta-llama/Meta-Llama-3-8B" # Ensure you have access on HF
model_id = "mistralai/Mistral-7B-Instruct-v0.3" #mistralai/Mistral-7B-Instruct-v0.3
output_dir = "./farming-llm-finetuned"

# 2. Data Preparation
dataset = load_dataset("argilla/farming", split="train")

def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        # Merging columns into a single 'text' block for next-token prediction
        text = f"### Instruction: {example['instruction'][i]}\n### Response: {example['response'][i]}"
        output_texts.append(text)
    return output_texts

# 3. Model & Tokenizer Setup (4-bit Quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 1. Force the model to load in 16-bit first (never 32-bit)
# 2. Use low_cpu_mem_usage to prevent RAM doubling
# 3. Use a pre-quantized or sharded version if available

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,     # CRITICAL: Prevents loading in float32
    low_cpu_mem_usage=True,        # CRITICAL: Optimizes System RAM usage
    trust_remote_code=True,
    # offload_folder="offload",    # Optional: Offloads to disk if RAM is tiny
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
# 4. PEFT (LoRA) Configuration

def get_trainable_params(model):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())

    perc = 100 * trainable_params / all_params

    print(f"Trainable: {trainable_params:,}")
    print(f"Total: {all_params:,}")
    print(f"Percentage: {perc:.4f}%")

    return trainable_params, all_params, perc

# Usage
trainable, total, percentage = get_trainable_params(model)

model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model.print_trainable_parameters()

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=1, # Start with 1 to avoid overfitting on small datasets
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
)

# 6. Initialize SFTTrainer
# SFTTrainer handles the 'text' column mapping and data collation automatically
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

# 7. Execute Training
trainer.train()

# 8. Save the results
trainer.save_model(output_dir)